In [0]:
import os
import requests
import json
from datetime import datetime, timezone
from pyspark.sql.functions import col, concat_ws, current_timestamp, lit
from pyspark.sql.types import ArrayType, FloatType
import pandas as pd
from pyspark.sql.functions import pandas_udf

# Get secrets from environment
api_key = os.environ.get("OPENAI_API_KEY")
endpoint = os.environ.get("OPENAI_ENDPOINT")
if api_key:
    api_key =api_key.strip("'")
if endpoint:
    endpoint =endpoint.strip("'")


In [0]:
run_id = dbutils.jobs.taskValues.get(
    taskKey="bronze_ingestion",
    key="run_id",
    debugValue="local-debug-run"
)
start_time = dbutils.jobs.taskValues.get(
    taskKey="bronze_ingestion",
    key="start_time",
    debugValue=str(datetime.now())
)
print(f"run_id: {run_id}")
print(f"start_time: {start_time}")

In [0]:


# Headers required for Azure OpenAI API
headers = {
    "Content-Type": "application/json",
    "api-key": api_key
}

# Define a Pandas UDF that returns an array of floats (embedding vector)
@pandas_udf(ArrayType(FloatType()))
def embed_batch(texts: pd.Series) -> pd.Series:
    
    # -------------------------------
    # Step 1: Clean input batch
    # -------------------------------
    # We create a new list that:
    # - keeps valid text as-is
    # - replaces null/empty text with None
    # This preserves original row structure
    input_texts = []
    
    for text in texts:
        if text is None or text.strip() == "":
            input_texts.append(None)   # mark invalid values
        else:
            input_texts.append(text)   # keep valid values
    
    # -------------------------------
    # Step 2: Extract only valid texts
    # -------------------------------
    # Remove None values → this is what we send to API
    valid_texts = [t for t in input_texts if t is not None]
    
    # Only call API if we actually have valid data
    if valid_texts:
        
        # Prepare request body (batch input)
        body = {
            "input": valid_texts
        }
        
        # Make API call
        response = requests.post(
            endpoint,
            headers=headers,
            json=body
        )
        
        # -------------------------------
        # Step 3: Parse API response
        # -------------------------------
        if response.status_code == 200:
            
            # Extract "data" field from response JSON
            data = response.json()["data"]
            
            # Extract embeddings from each item
            # Each item is like: {"embedding": [...]}
            valid_embeddings = [item["embedding"] for item in data]
        
        else:
            # If API fails, create fallback list of None
            # (same length as valid_texts to preserve alignment)
            valid_embeddings = [None] * len(valid_texts)
    
    else:
        # If no valid input at all → no embeddings
        valid_embeddings = []
    
    # -------------------------------
    # Step 4: Rebuild original structure
    # -------------------------------
    # We now map embeddings back to original rows
    
    result = []
    valid_ptr = 0  # pointer to track embeddings list
    
    for text in input_texts:
        if text is None:
            # Original value was invalid → keep None
            result.append(None)
        else:
            # Take next embedding from valid_embeddings
            result.append(valid_embeddings[valid_ptr])
            valid_ptr += 1  # move pointer forward
    
    # -------------------------------
    # Step 5: Return as Pandas Series
    # -------------------------------
    # Spark will convert this into a column (array<float>)
    return pd.Series(result)

In [0]:
try:
    from pyspark.sql import functions as F
    silver_df = spark.sql("""
                          SELECT url, title, description,publishedAt, __START_AT FROM news_pipeline.silver.silver_articles where title IS NOT NULL
                          AND __END_AT IS NULL 
                          """)
    silver_df = silver_df.withColumn("text",concat_ws(" ",col("title"),col("description")))

    # Register temp views
    silver_df.createOrReplaceTempView("silver_view")

    gold_exists = True
    try:
        spark.sql("SELECT 1 FROM news_pipeline.gold.gold_articles_embeddings LIMIT 1")
    except:
        gold_exists = False

    if gold_exists:
        spark.sql("""
            CREATE OR REPLACE TEMP VIEW new_rows AS
            SELECT s.*
            FROM silver_view s
            LEFT ANTI JOIN news_pipeline.gold.gold_articles_embeddings g
            ON s.url = g.url and s.__START_AT = g.__START_AT
        """)
        new_df = spark.table("new_rows")
    else:
        new_df = silver_df

    result_df = new_df.withColumn(
        "embedding",
        embed_batch("text")
    ).withColumn("ingested_at",current_timestamp())


    mode = "overwrite" if not gold_exists else "append"

    result_df.write \
        .format("delta") \
        .mode(mode) \
        .saveAsTable("news_pipeline.gold.gold_articles_embeddings")

    end_time = datetime.now()
    start_dt = datetime.strptime(start_time, "%Y-%m-%d %H:%M:%S.%f")
    duration_spent = (end_time - start_dt).total_seconds()

    silver_records = int(spark.sql("SELECT COUNT(*) FROM news_pipeline.silver.silver_articles").collect()[0][0])
    
    gold_rows = int(
        spark.sql("""
            SELECT COUNT(*)
            FROM news_pipeline.gold.gold_articles_embeddings
        """).collect()[0][0]
    )
    spark.sql(f"""
        UPDATE news_pipeline.monitoring.pipeline_runs
        SET 
            gold_records = {gold_rows},
            silver_records = {silver_records},
            end_time = CAST('{end_time}' AS TIMESTAMP),
            duration_spent = {duration_spent},
            status = 'succeeded'
        WHERE run_id = '{run_id}'
    """)
except Exception as e:
    try:
        end_time = datetime.now()
        spark.sql(f"""
            UPDATE news_pipeline.monitoring.pipeline_runs
            SET 
                status = 'failed',
                end_time = CAST('{end_time}' AS TIMESTAMP),
                error_message = '{str(e)[:500]}'
            WHERE run_id = '{run_id}'
        """)
    except:
        pass
    raise e